### ESA_maap_dlpar.ipynb minimum requirements
- credentials.txt (token=以降にtokenを設定)  
上記配置場所をMAAP_CREDで指定する必要  
- In[1] (最初のセル)での以下の指定
1. 取得期間②対象プロダクトとバージョン
2. 1.に対応したカタログ
3. ダウンロードディレクトリ

詳細はREADME.mdを参照のこと

```text
Copyright (c) 2026 JAXA/SAOC, Kotaro R. Araragi  
Released under the MIT License.  
https://opensource.org/licenses/MIT  
```




In [36]:
### Data loading setup
START_date="2026-08-01T00:00:00Z"
END_date  ="2026-08-05T23:59:59Z"

#取得プロダクトとバージョン
target_prd ="ACM_CAP_2B" #10 char
target_var="CA" #see at https://earth.esa.int/eogateway/missions/earthcare/data
FLG_VAR_LATEST=1 # 0 : just select target_var, 1 : automatically select all versions

#EC_COLLECTION = ['EarthCAREL1Validated_MAAP']
EC_COLLECTION  = ['EarthCAREL2Validated_MAAP']
# ESA MAAP catalogue:
# EarthCAREL1Validated_MAAP (CPR L1b available)
# EarthCAREL2Validated_MAAP, JAXAL2Validated_MAAP

#Colabの場合 /content/drive/MyDrive以下を指定
WORK_DIR="/content/drive/MyDrive"
DL_DIR=WORK_DIR+"/EC_DL" #download directory
DL_URL=WORK_DIR+"/EC_URL" #url text,GURL_FLG=0なら不要

MAAP_CRED=WORK_DIR+"/MAAP/credentials.txt" #token



In [ ]:
import requests
import glob
import os, time, sys
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import shutil #ファイル移動

#日付取得
from datetime import datetime,date, timedelta #date, timedeltaは後で使う
strt_date = datetime.strptime(START_date, "%Y-%m-%dT%H:%M:%SZ").date()
end__date = datetime.strptime(END_date, "%Y-%m-%dT%H:%M:%SZ").date()
DDAYS = (end__date - strt_date).days + 1


In [38]:
#各定数, フラグ
### ESA MAAP (STAC) Catalog API
#Point to the catalogue endpoint

avail_cpus=8 #並列取得数Colab Proまでなら8程度が妥当。
drive_limit=85 #google driveの停止水準(%)

NOGET_FLG=0 #1なら処理を停止。urlファイルのみ
FLG_HDR=0 #1ならHDRファイルまで取得
GURL_FLG=0 #1なら取得したurlを残す。 0なら削除

catalog_url = 'https://catalog.maap.eo.esa.int/catalogue/'
TARGET_ID = "EX"+str(target_var).upper()+"_"+target_prd


In [25]:
#filesはfiles.download("/content/download.zip")などでDownload可能
try:
    from google.colab import drive, files
except ImportError:
    USE_COLAB = False
else:
    USE_COLAB = True
    #pystac_clientを使用
    !pip3 install pystac_client

if USE_COLAB:
  GDRIVE_DIR="/content/drive"
  drive.mount(GDRIVE_DIR)

DL_ESA_Prd = Path(DL_DIR)
#ディレクトリがなければ作成
DL_ESA_Prd.mkdir(parents=False, exist_ok=True)

if GURL_FLG==1:
  DL_ESA_urls = Path(DL_URL)
  DL_ESA_urls.mkdir(parents=False, exist_ok=True)

#前回実行分を削除
for path in Path.cwd().glob("urls_*.txt"):
    if path.is_file():
        path.unlink()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
#マウントしたdriveの容量を取得

if USE_COLAB:
  #httplib2の警告抑止用。なくてもよい
  import logging
  logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)

  from google.colab import auth
  auth.authenticate_user()
  from googleapiclient.discovery import build
  g_service = build('drive', 'v3')

else:
  #ローカル環境
  g_service = None

#driveの容量を見てファイルを停止する
def check_quota(service,output_dir):
    if service is None:
        usage = shutil.disk_usage(str(output_dir))
        return usage.used, usage.total

    quota = service.about().get(fields="storageQuota").execute()["storageQuota"]
    used = int(quota["usage"])
    raw_limit = quota.get("limit")
    limit = int(raw_limit) if raw_limit is not None else None

    return used, limit

# used, limitで警告や停止を実施
def evaluate_quota(used, limit, PRINT_FLG=1,SYS_FLG=1):
  #PRINT_FLG=1 : 1 なら容量表示
  #SYS_FLG=1 : 1は
  drive_warning = 85 #警告の水準/停止措置はなし
  rate_percent=100*used/limit
  if PRINT_FLG==1:
    print(f"Rate : {rate_percent:.1f}%, Used : {used/1024**3:.2f} GB / Total {limit/1024**3:.2f} GB")

  if rate_percent>=drive_warning:
    print("Disk usage warning", str(rate_percent),">=",str(drive_warning), "%")

  if rate_percent>=drive_limit:
    print("Disk usage exceed", str(drive_limit), "%")
    if SYS_FLG==1:
      input("空き容量不足, このまま実施しますか？(Enter)")
    else:
      sys.exit("処理停止。取得ファイル作成のみ")

  return


used_ini, limit_ini = check_quota(g_service,DL_DIR)
evaluate_quota(used_ini,limit_ini)


Rate : 0.5%, Used : 0.49 GB / Total 100.00 GB


In [ ]:
#%%time
###大容量ファイル向け
### pystac_clientとpystacが利用の場合
from pystac_client import Client
#from pystac import ItemCollection
catalog = Client.open(catalog_url)

items = []
d = strt_date
retry_catalog=5 #retry用。ファイルアクセスが多いとリトライが必要な場合がある
while d <= end__date: #d - d+1なので

    next_d = d + timedelta(days=1)

    for retry in range(retry_catalog):

        try:
            #フレーム指定の場合はここを指定
            search = catalog.search(
                collections=EC_COLLECTION,
                filter=f"productType = '{target_prd}'",
                #datetime=[d.isoformat(), next_d.isoformat()],
                datetime=[d.isoformat(), d.isoformat()],
                method="GET",
                limit=1200      # 124*4以上でも可
            )


            for page in search.pages_as_dicts():
                for feature in page["features"]:
                    items.append(
                        (
                            feature["id"],
                            feature["assets"]["enclosure_h5"]["href"]
                        )
                    )

            break

        except Exception as e:
            print(
                f"{d}: retry {retry+1}/5 failed: {e}"
            )
            if retry == (retry_catalog-1):
                raise
            time.sleep(2**retry)

    d = next_d

items.sort(key=lambda x: x[0])
#重複を取る。日をまたぐところは重複がでる
items = list(dict.fromkeys(items))

#collection = ItemCollection(data2)
#data_out = collection.to_dict()

In [42]:
print(len(items), "files selected")
product_ids = [product_id for product_id, url in items]
#print(product_ids)
prefix_types = sorted({product_id[6:8] for product_id in product_ids})
print(len(prefix_types)," available versions",", ".join(prefix_types))
#バージョン最新のものだけ選択
last_initial = max((code[0] for code in prefix_types), default="")
prefix_types_selected = [code for code in prefix_types if code.startswith(last_initial)]

609 files selected
1  available versions BC


In [29]:
#認証情報読み込み
# from https://portal.maap.eo.esa.int/ini/kb/books/collaborative-environment-pal/page/earthcare-maap

# --- Path to credentials.txt ---
CREDENTIALS_FILE = Path(MAAP_CRED)   # Insert the .txt path

def load_credentials(file_path=CREDENTIALS_FILE):
    """Read key-value pairs from a credentials file into a dictionary."""
    creds = {}
    if not file_path.exists():
        raise FileNotFoundError(f"Credentials file not found: {file_path}")
    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if "=" not in line:
                continue
            key, value = line.split("=", 1)
            creds[key.strip()] = value.strip()
    return creds


# --- ESA MAAP API ---

def get_token():
    """Use OFFLINE_TOKEN to fetch a short-lived access token."""
    creds = load_credentials()

    OFFLINE_TOKEN = creds.get("OFFLINE_TOKEN")
    CLIENT_ID = creds.get("CLIENT_ID")
    CLIENT_SECRET = creds.get("CLIENT_SECRET")
    print(CLIENT_SECRET)

    if not all([OFFLINE_TOKEN, CLIENT_ID, CLIENT_SECRET]):
        raise ValueError("Missing OFFLINE_TOKEN, CLIENT_ID, or CLIENT_SECRET in credentials file")

    url = "https://iam.maap.eo.esa.int/realms/esa-maap/protocol/openid-connect/token"
    data = {
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "grant_type": "refresh_token",
        "refresh_token": OFFLINE_TOKEN,
        "scope": "offline_access openid"
    }

    response = requests.post(url, data=data)
    response.raise_for_status()

    response_json = response.json()
    access_token = response_json.get('access_token')

    if not access_token:
        raise RuntimeError("Failed to retrieve access token from IAM response")

    return access_token

In [30]:
#リストを変換
ds_url_list = [url for _, url in items]
#print(ds_url_list)
#urlを文字列に変換。
ds_url_list_check = [str(x).strip() for x in ds_url_list if x is not None]
#全バージョンを保持
ds_url_list_par = list(dict.fromkeys(ds_url_list_check))
#print(ds_url_list_par)

with open("urls_all.txt","w") as f:
    for url in ds_url_list_par:
        f.write(url+"\n")

In [ ]:
#tokenの取得
token = get_token()
TOKEN = token

p1eL7uonXs6MDxtGbgKdPVRAmnGxHpVE


In [ ]:

#バージョン適合のみ抽出
if FLG_VAR_LATEST==0:
  with open("urls_all.txt") as f:
      exbc_urls = [line for line in f if TARGET_ID in line]
  print("Choose: ",TARGET_ID)
  print("select ",str(len(exbc_urls))," from ",str(len(ds_url_list_par)))
  if len(exbc_urls) == 0:
    sys.exit("no files:"+str(TARGET_ID) )

else:
#最新バージョンの選択
  print(target_prd,len(prefix_types)," available versions",", ".join(prefix_types))
  print("select : ",str(len(ds_url_list_par)), "files")
  with open("urls_all.txt") as f:
      exbc_urls = [line for line in f]

with open("urls.txt", "w") as f:
    f.writelines(exbc_urls)

#中間出力を削除
Path("urls_all.txt").unlink()


ACM_CLP_2B 1  available versions BB
select :  610 files


In [ ]:
import re

#from text file
def make_hdr_url_file(src: Path) -> Path:
    dst = src.with_name(f"{src.stem}_hdr{src.suffix}")
    dst.write_text(
        re.sub(r"\.h5\b", ".HDR", src.read_text(encoding="utf-8")),
        encoding="utf-8",
    )
    return dst

def make_hdr_url_txt(src_line):
    dst = str(Path(src_line).with_suffix(".HDR"))
    return dst


In [ ]:
#from pathlib import Path
#import re
from collections import defaultdict

#バージョンを取り出したテキストファイルをさらに日付別にする
input_file = Path("urls.txt")

# 日付ごとのURL格納用
url_dict = defaultdict(list)

#ここを格納
#TARGET_ID = "EX??_ATL_NOM_1B"
if FLG_VAR_LATEST==0:
  pattern = re.compile(
      rf"{re.escape(TARGET_ID)}_(\d{{8}})T\d+"
  )
else:
  pattern = re.compile(
      #rf"{re.escape(target_prd)}_(\d{{8}})T\d+" #target_idのみ一致の場合
      rf"ECA_[E,J]X{re.escape(last_initial)}[A-Z]_{re.escape(target_prd)}_(\d{{8}})T\d+"
  )


with open(input_file, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        m = pattern.search(line)
        if m:
            yyyymmdd = m.group(1)
            url_dict[yyyymmdd].append(line)

# ファイル出力
i_str=0 #最初のファイル用
for yyyymmdd, urls in url_dict.items():
    out_file = Path(f"urls_{yyyymmdd}.txt")
    with open(out_file, "w", encoding="utf-8") as f:
        f.write("\n".join(urls))
        f.write("\n")
    #最初は時刻のずれで日またぎ1つなのでマージ
    if i_str==0:
        merge=out_file
        i_str+=1
    elif i_str==1:
        #2番目に結合
        print("Merge ",merge.name, " and ",out_file.name)
        merge_text = merge.read_text()
        out_text = out_file.read_text()

        out_file.write_text(merge_text + out_text)
        merge.unlink()
        i_str+=1

print(f"{len(url_dict)} files created.")

#中間出力を削除
Path("urls.txt").unlink()


In [ ]:
#urlファイル作成だけならここまで
if NOGET_FLG == 1:
    sys.exit("NOGET_FLG=1 のため処理停止。取得ファイル作成のみ")


In [ ]:
#ハッシュ値計算
import hashlib

def append_tar_md5(tar_name):
    tar_path = Path(tar_name)
    md5 = hashlib.md5()

    with tar_path.open("rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            md5.update(chunk)

    hash_value = md5.hexdigest()

    # 固定のテキストファイルに追記（保存先は適宜変更）
    with open("tar_md5.txt", "a", encoding="utf-8") as f:
        f.write(f"{tar_path.name} {hash_value}\n")

    return hash_value

os.environ['TOKEN'] = TOKEN
headers = {"Authorization": f"Bearer {TOKEN}"}

MAX_RETRY = 5 #通常5回程度で十分

def download(url):
    filename = url.split("/")[-1]

    for attempt in range(1, MAX_RETRY + 1):
        try:
            with requests.get(
                url,
                headers=headers,
                stream=True,
                timeout=60
            ) as r:
                r.raise_for_status()

                with open(filename, "wb") as f:
                    for chunk in r.iter_content(1024 * 1024):
                        if chunk:
                            f.write(chunk)

            print(f"OK {filename}")
            #移動
            filename_mid=filename+".tmp" #コピー中を識別するため使用
            shutil.move(str(filename), DL_ESA_Prd / str(filename_mid) )
            shutil.move(DL_ESA_Prd / str(filename_mid), DL_ESA_Prd / str(filename) )

            return True

        except Exception as e:
            print(
                f"[{attempt}/{MAX_RETRY}] "
                f"{filename} failed: {e}"
            )

            # 中途半端なファイルを削除
            try:
                Path(filename).unlink()
            except FileNotFoundError:
                pass

            if attempt < MAX_RETRY:
                time.sleep(5)

    print(f"NG ############################################################")
    print(f"NG {filename}")
    print(f"NG ############################################################")
    return False
#################################################

In [ ]:
#日付別に取得実施

#ディスクチェックの制御
MAX_REQ_PER_SEC = 2 #秒毎の呼び出し制限
last_call = 0 #呼び出し秒数

t_str=time.time()
# DDAYS=(end_date - start_date).days + 1
for i in range(DDAYS):
    iday = strt_date + timedelta(days=i)
    iDATE= iday.strftime("%Y%m%d")
    print("loading ",iDATE)
##################################################
##################################################
#以下インデント

    #iDATE=20250112
    delete=0
    itxt="urls_"+str(iDATE)+".txt"
    urls = Path(itxt).read_text().splitlines()
    #uFLG_HDR 1でHDRも取得
    if FLG_HDR==1:
        urls.extend([
            str(Path(line).with_suffix(".HDR"))
            for line in urls
            if Path(line).suffix == ".h5"
        ])

        print("##############################")
        print("with HDR:")
        #print(urls)
        print("##############################")

    #容量をDL前に確認
    elapsed = time.time() - last_call
    if elapsed < 1 / MAX_REQ_PER_SEC:
        time.sleep(1 / MAX_REQ_PER_SEC - elapsed)
    used_int, limit_int = check_quota(g_service, DL_DIR)
    evaluate_quota(used_int,limit_int)
    last_call = time.time()

    #itxtの1ファイル内のh5のみDL
    t_int_pre=time.time() #download前の時間
    with ThreadPoolExecutor(max_workers=avail_cpus) as executor:
        #executor.map(download, urls) #全スレッドを待たない場合
        list(executor.map(download,urls )) #listは全スレッドを待つ場合

    if GURL_FLG==1:
      shutil.move(str(itxt),DL_ESA_urls / str(itxt))
    else:
      Path(itxt).unlink()

    t_end=time.time()
    print("SAVE at",DL_ESA_Prd," ",str(t_end-t_int_pre)," sec" )
    print("total ",t_end-t_str," sec")
    # カレントディレクトリ配下の .tar ファイルを移動
    #for tarfile in Path(".").glob("*.tar"):
    #    shutil.move(str(tarfile), DL_ESA_Prd / tarfile.name)

##################################################
##################################################

print("all process end")
t_end=time.time()
print(t_end-t_str," sec")

